# Skip edges

`parse_edgelist()` layers a DAG. Edges with depth gap exactly 1
become `GraphSpec.masks`. An original edge whose endpoints are more
than one layer apart is a **skip**. Skips are recorded in
`spec.skips` only. They are not written into the masks and are not
expanded into dummy neurons.

`MaskedLinear` remains one adjacent hop. `SkipAdd` is a separate
`nn.Module` that injects skip sources into the **target
pre-activation**. The user still owns `forward()`, ReLU, BatchNorm,
and the rest.

The [Getting started](../getting-started/) notebook uses a graph
with no skips. This page shows how to wire graphs that jump a
layer with `SkipAdd`.


## What a skip edge means

An adjacent chain such as `H1 -> H2` is a normal layer: one
`MaskedLinear` maps `layer_nodes[i]` to `layer_nodes[i + 1]`.

A skip such as `A -> H2` is still a directed edge into `H2`. `A` is
an extra **parent of the target unit**. It skips every layer between
source and target. It does not skip the target. The source is not
copied through dummy units, and it is not added onto the tensor
*after* the target nonlinearity.

`kpnn2` does not insert identity neurons to turn a skip into a
chain of adjacent hops. There is no generated node name.
`MaskedLinear` never stores other layers' activations. Completing
the incoming mix is `SkipAdd`'s job.

![SkipAdd](figures/skip_add.svg)

**Figure 1.** Adjacent edges occupy `spec.masks`. Skip edges
`A -> H2`, `H1 -> C`, and `A -> C` are recorded in `spec.skips`
only. `SkipAdd` adds each source into the target pre-activation
(after that layer's `MaskedLinear`, before ReLU or BatchNorm).
Source activations are not modified.

Construct `SkipAdd` once from the full `GraphSpec`. It holds one
learnable scalar per skip. Call it after every hop. If no skip
targets that layer, the call is a no-op. A skip into a hidden layer
and a skip into the output use the same mechanism.

**Call order.** After each hop: `MaskedLinear`, then `SkipAdd` for
that target layer, then ReLU, BatchNorm, or dropout. Those later
modules see one layer tensor that already includes the skip terms.
`A` does not pass through the adjacent weight matrix. It does share
everything applied *after* the injection.

In symbols, for target `H2` with adjacent parent `H1` and skip
parent `A`:

```text
H2 = relu( w_{H1→H2} H1 + b_{H2} + w_{A→H2} A )
```

There is no extra skip bias: one bias per unit stays on
`MaskedLinear`. One weight per incoming edge: adjacent weights in
the mask, skip weights on `SkipAdd`.

`spec.masks` never contain skip edges. `map_node_attributions()`
labels original `layer_nodes` names. If a skip affects the
prediction, that effect appears on the named target unit, not on a
dummy channel.


## A three-hop module

The example below is the graph in Figure 1: inputs `A` and `B`,
hidden units `H1` and `H2`, output `C`. Adjacent edges are `A -> H1`,
`B -> H1`, `H1 -> H2`, and `H2 -> C`. Skips are `A -> H2`, `H1 -> C`,
and `A -> C`.


In [1]:
import pandas as pd

import kpnn2 as k2

edgelist = pd.DataFrame(
    {
        "source": ["A", "B", "H1", "H2", "A", "H1", "A"],
        "target": ["H1", "H1", "H2", "C", "H2", "C", "C"],
    }
)
spec = k2.parse_edgelist(edgelist)
spec.layer_nodes, spec.skips

((('A', 'B'), ('H1',), ('H2',), ('C',)),
 (Skip(source='A', target='H2', source_layer=0, target_layer=2, source_index=0, target_index=0),
  Skip(source='H1', target='C', source_layer=1, target_layer=3, source_index=0, target_index=0),
  Skip(source='A', target='C', source_layer=0, target_layer=3, source_index=0, target_index=0)))

In [2]:
mask_shapes = [
    tuple(mask.shape)
    for mask in spec.masks
]
n_adjacent = int(
    sum(
        mask.sum().item()
        for mask in spec.masks
    )
)
mask_shapes, n_adjacent

([(1, 2), (1, 1), (1, 1)], 4)

Each mask is `(n_{i+1}, n_i)` with a `1.0` only for an original
adjacent edge. The four ones are the adjacent hops. The three skip
edges are in `spec.skips` only.

The module uses one `MaskedLinear` per hop and one `SkipAdd` for
the whole spec. `SkipAdd` is registered once in `__init__`. The
sequence of injections is the call order in `forward()`.


In [3]:
import torch
import torch.nn.functional as F
from torch import nn


class Net(nn.Module):
    def __init__(
        self,
        spec: k2.GraphSpec,
    ):
        super().__init__()
        self.lin0 = k2.MaskedLinear(
            spec.masks[0],
            bias=False,
        )
        self.lin1 = k2.MaskedLinear(
            spec.masks[1],
            bias=False,
        )
        self.lin2 = k2.MaskedLinear(
            spec.masks[2],
            bias=False,
        )
        self.skips = k2.SkipAdd(spec)

    def forward(
        self,
        x,
    ):
        saved = {0: x}

        h1 = self.lin0(x)
        h1 = self.skips(
            h1,
            saved,
            target_layer=1,
        )
        h1 = F.relu(h1)
        saved[1] = h1

        h2 = self.lin1(h1)
        h2 = self.skips(
            h2,
            saved,
            target_layer=2,
        )
        h2 = F.relu(h2)
        saved[2] = h2

        out = self.lin2(h2)
        out = self.skips(
            out,
            saved,
            target_layer=3,
        )
        saved[3] = out
        self.saved = saved
        return out


model = Net(spec)

The last hop has no ReLU, which is a modeling choice. `SkipAdd`
still runs, so skips into `C` join `C`'s pre-activation.

A skip cannot target layer 1 under longest-path layering (the depth
gap would not exceed 1). The `target_layer=1` call is nevertheless
part of the uniform template.

After a forward pass, `model.saved` holds the layer tensors: input
at `0`, post-ReLU `H1` and `H2` at `1` and `2`, and `C` at `3`.

### Numerical check

Pin every live adjacent weight to `1`, disable bias, and set the
skip scalars to `w_{A→H2} = 0.3`, `w_{H1→C} = 0.2`,
`w_{A→C} = 0.1`. For input `A = 2`, `B = 0`:

```text
H1 = relu(A + B) = 2
H2 = relu(H1 + 0.3 A) = relu(2.6) = 2.6
C  = H2 + 0.2 H1 + 0.1 A = 3.2
```

For `A = -1`, `B = 0`, ReLU zeros the adjacent path through `H1`
and `H2`, but the skip `A -> C` still contributes:

```text
H1 = relu(-1) = 0
H2 = relu(0 + 0.3 (-1)) = 0
C  = 0 + 0.2 * 0 + 0.1 * (-1) = -0.1
```

That extra term is `w *` the saved source, not a dummy unit in
layer 1 or 2.

In [4]:
skip_values = {
    ("A", "H2"): 0.3,
    ("H1", "C"): 0.2,
    ("A", "C"): 0.1,
}
with torch.no_grad():
    for layer in (
        model.lin0,
        model.lin1,
        model.lin2,
    ):
        layer.raw_weight.copy_(layer.mask)
    for skip, weight in zip(
        spec.skips,
        model.skips.skip_weights,
        strict=True,
    ):
        weight.fill_(
            skip_values[(skip.source, skip.target)]
        )


def layer_values(x):
    with torch.no_grad():
        y = model(x)
        return {
            "H1": model.saved[1],
            "H2": model.saved[2],
            "C": y,
        }


x_pos = torch.tensor(
    [[2.0, 0.0]],
)
x_neg = torch.tensor(
    [[-1.0, 0.0]],
)
layer_values(x_pos), layer_values(x_neg)

({'H1': tensor([[2.]]), 'H2': tensor([[2.6000]]), 'C': tensor([[3.2000]])},
 {'H1': tensor([[0.]]), 'H2': tensor([[0.]]), 'C': tensor([[-0.1000]])})

## Step by step

The walkthrough follows the `forward()` listing above. `x` has one
row per sample and columns `(A, B)` in `spec.input_nodes` order.

**Save layer 0.** `saved = {0: x}` keeps the input tensor. Later
skips `A -> H2` and `A -> C` read the `A` column from this entry.
`A` is never overwritten.

**Hop 0.** `h1 = self.lin0(x)` is the adjacent map from `(A, B)` to
`H1` (`spec.masks[0]`). This is the `H1` pre-activation from
adjacent parents only.

**Skips into layer 1.** `self.skips(..., target_layer=1)` finds no
skip with that target (a skip needs a depth gap greater than 1, so
the earliest possible target is layer 2). `h1` is unchanged.

**Nonlinearity and stash.** `h1 = F.relu(h1)` applies ReLU to `H1`.
`saved[1] = h1` stores the finished layer-1 tensor for `H1 -> C`.

**Hop 1.** `h2 = self.lin1(h1)` is the adjacent map `H1 -> H2`.
`A` is not a column of this layer. If ReLU zeroed `H1`, this
adjacent pre-activation is 0 so far.

**Skips into layer 2.** `A -> H2` matches `target_layer=2`. `SkipAdd`
reads `A` from `saved[0]`, multiplies by that skip's scalar, and
adds the result into the `H2` slot. `A` is still the original
input value. The adjacent `H1 -> H2` term is unchanged.

```text
H2_pre = (adjacent from H1) + w_{A→H2} * A
```

**Nonlinearity and stash.** `h2 = F.relu(h2)` applies ReLU to the
**sum**, so the skip term is included in `H2`'s nonlinearity.
`saved[2] = h2` stores the finished layer-2 tensor. This example
has no skip that reads `H2`, but the template always saves.

**Hop 2.** `out = self.lin2(h2)` is the adjacent map `H2 -> C`.

**Skips into layer 3.** `H1 -> C` and `A -> C` both match.
`SkipAdd` adds `w_{H1→C} * H1` using the already-ReLU'd `saved[1]`,
and `w_{A→C} * A` using `saved[0]`:

```text
C = (adjacent from H2) + w_{H1→C} * H1 + w_{A→C} * A
```

There is no ReLU after this hop in the listing, so those skip terms
are not passed through a further nonlinearity. The return value is
`C`.

Each `SkipAdd` call returns a new tensor: the incoming layer plus
any matching skip terms. Units with no incoming skip are unchanged
by that call. The source tensors in `saved` stay as they were when
that layer was finished.
